# Kolmogorov-Arnold Sparse Attention (KASA) for Wheat Futures

**Three-step progressive upgrade of the attention bottleneck in `CrossAttentionLSTM` and `DualAttentionLSTM`:**

| Step | Change | Goal |
|------|--------|------|
| **Step 1** | Sparsemax replaces Softmax in `AdditiveAttention` | Noise elimination via exact-zero weights |
| **Step 2** | KAN B-splines replace linear layers for score computation | Non-linear similarity on concatenated 80-D state |
| **Step 3** | Full KASA = KAN splines + Sparsemax + asymmetric bias | Synthesis: sparse + non-linear routing |

Each step is evaluated with 5-Fold Time-Series CV and compared against the `BaselineModels_WheatFutures.ipynb` baselines:
```
Cross-Attention:   Acc=0.5133 | F1=0.4561 | Recall(Up)=0.4305
Dual-Attention:    Acc=0.5353 | F1=0.3884 | Recall(Up)=0.3113
```

## 1. Setup & Imports

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'entmax', '-q'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, f1_score)
import random, warnings
import seaborn as sns
from entmax import sparsemax

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
print('entmax sparsemax imported successfully.')

## 2. Load & Prepare Data

Identical pipeline to `BaselineModels_WheatFutures.ipynb` — price features + FinBERT 16D PCA embeddings + decayed sentiment.

In [ ]:
base_path = 'data'

price_df = pd.read_csv(f'{base_path}/wheat_prices.csv')
price_df['Date'] = pd.to_datetime(price_df['Date'])
price_df = price_df.sort_values('Date').set_index('Date')

for col in ['Price', 'Open', 'High', 'Low']:
    price_df[col] = price_df[col].replace({',': ''}, regex=True).astype(float)

def parse_volume(v):
    if pd.isna(v) or str(v).strip() in ('', '-'): return np.nan
    v = str(v).strip().replace(',', '')
    if v.endswith('K'): return float(v[:-1]) * 1_000
    elif v.endswith('M'): return float(v[:-1]) * 1_000_000
    return float(v)

price_df['Volume'] = price_df['Vol.'].apply(parse_volume)

sentiment_df = pd.read_csv(f'{base_path}/daily_news_sentiment.csv')
sentiment_map = dict(zip(sentiment_df['date'], sentiment_df['sentiment_score']))

news_emb = torch.load(f'{base_path}/daily_news_embeddings.pt',
                      map_location='cpu', weights_only=False)
dates_with_news_set = set(news_emb.keys())

print(f'Price data: {len(price_df)} trading days')
print(f'Sentiment dates: {len(sentiment_map)} | News embedding dates: {len(dates_with_news_set)}')

In [ ]:
# ── Technical Indicators ──────────────────────────────────────────────────────
price_df['Return']     = np.log(price_df['Price'] / price_df['Price'].shift(1))
price_df['Volatility'] = price_df['Return'].rolling(5).std()

delta = price_df['Price'].diff()
gain  = delta.where(delta > 0, 0.0).rolling(14).mean()
loss  = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
price_df['RSI'] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

ema12 = price_df['Price'].ewm(span=12, adjust=False).mean()
ema26 = price_df['Price'].ewm(span=26, adjust=False).mean()
price_df['MACD'] = ema12 - ema26

bb_mid = price_df['Price'].rolling(20).mean()
bb_std = price_df['Price'].rolling(20).std()
price_df['BB_pctB'] = (price_df['Price'] - (bb_mid - 2*bb_std)) / (4*bb_std)

price_df['Volume'] = price_df['Volume'].ffill().bfill()
vol_chg = price_df['Volume'].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
price_df['Vol_chg'] = vol_chg

# ── Decayed Sentiment ────────────────────────────────────────────────────────
HALF_LIFE_DAYS = 3
decay_rate = np.log(2) / HALF_LIFE_DAYS
sentiment_dates = sorted(sentiment_map.keys())

def get_decayed_sentiment(date):
    query_date = (date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    best_date = None
    for sd in sentiment_dates:
        if sd <= query_date: best_date = sd
        else: break
    if best_date is None: return 0.0
    days_since = (pd.to_datetime(query_date) - pd.to_datetime(best_date)).days
    return sentiment_map[best_date] * np.exp(-decay_rate * days_since)

price_df['Sentiment'] = [get_decayed_sentiment(d) for d in price_df.index]
price_df['Target']    = (price_df['Price'].shift(-1) > price_df['Price']).astype(int)

price_df.replace([np.inf, -np.inf], np.nan, inplace=True)
price_df = price_df.dropna(subset=['Return','Volatility','RSI','MACD','BB_pctB','Target'])

PRICE_SENT_FEATURES = ['Price','Return','Volatility','RSI','MACD','BB_pctB','Vol_chg','Sentiment']
print(f'Final dataset: {len(price_df)} trading days')
print(f'Target balance: Up={price_df["Target"].mean():.2%}')

In [ ]:
LOOKBACK = 30
EMB_DIM  = 16

n = len(price_df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)
train_df  = price_df.iloc[:train_end]
val_df    = price_df.iloc[train_end:val_end]
test_df   = price_df.iloc[val_end:]

def create_sequences(df, feature_cols, lookback, news_embeddings=None):
    features = df[feature_cols].values
    targets  = df['Target'].values
    date_idx = df.index.strftime('%Y-%m-%d').tolist()
    X_num, X_text, X_mask, y, seq_dates = [], [], [], [], []
    for i in range(len(df) - lookback):
        X_num.append(features[i:i + lookback])
        y.append(targets[i + lookback])
        seq_dates.append(date_idx[i + lookback])
        if news_embeddings is not None:
            text_seq, mask_seq = [], []
            for d in date_idx[i:i + lookback]:
                lag_d = (pd.to_datetime(d) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
                if lag_d in news_embeddings:
                    text_seq.append(news_embeddings[lag_d].numpy())
                    mask_seq.append(1.0)
                else:
                    text_seq.append(np.zeros(EMB_DIM))
                    mask_seq.append(0.0)
            X_text.append(text_seq)
            X_mask.append(mask_seq)
    X_num = np.array(X_num)
    y     = np.array(y)
    if news_embeddings is not None:
        return X_num, np.array(X_text), np.array(X_mask), y, seq_dates
    return X_num, np.zeros((len(X_num),lookback,EMB_DIM)), np.zeros((len(X_num),lookback)), y, seq_dates

X_full_all, X_text_all, X_mask_all, y_full_all, dates_full_all = create_sequences(
    price_df, PRICE_SENT_FEATURES, LOOKBACK, news_embeddings=news_emb)

train_cutoff = train_df.index.max().strftime('%Y-%m-%d')
val_cutoff   = val_df.index.max().strftime('%Y-%m-%d')

tr_idx = np.array([i for i, d in enumerate(dates_full_all) if d <= train_cutoff])
vl_idx = np.array([i for i, d in enumerate(dates_full_all) if train_cutoff < d <= val_cutoff])
te_idx = np.array([i for i, d in enumerate(dates_full_all) if d > val_cutoff])

print(f'Full sequences: X_num={X_full_all.shape}, X_text={X_text_all.shape}')
print(f'Split -> train={len(tr_idx)}, val={len(vl_idx)}, test={len(te_idx)}')

## 3. Shared Utilities

In [ ]:
# ── Baselines from BaselineModels_WheatFutures.ipynb (test set) ──────────────
BASELINES = {
    'Cross-Attention (baseline)': {'acc': 0.5133, 'f1': 0.4561, 'rec_up': 0.4305},
    'Dual-Attention (baseline)':  {'acc': 0.5353, 'f1': 0.3884, 'rec_up': 0.3113},
}

def get_activation(name):
    return {'relu': nn.ReLU, 'gelu': nn.GELU, 'mish': nn.Mish}.get(name, nn.ReLU)()

class GatedSkip(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.map  = nn.Linear(input_dim, hidden_dim)
        self.gate = nn.Linear(hidden_dim * 2, hidden_dim)
    def forward(self, context, skip_feat):
        s = self.map(skip_feat)
        g = torch.sigmoid(self.gate(torch.cat([context, s], dim=1)))
        return g * context + (1 - g) * s

def evaluate_predictions(y_true, y_pred_prob):
    y_pred = (y_pred_prob > 0.5).astype(int)
    return (y_pred,
            accuracy_score(y_true, y_pred),
            precision_score(y_true, y_pred, pos_label=1, zero_division=0),
            precision_score(y_true, y_pred, pos_label=0, zero_division=0),
            recall_score(y_true, y_pred, pos_label=1, zero_division=0),
            recall_score(y_true, y_pred, pos_label=0, zero_division=0),
            f1_score(y_true, y_pred),
            confusion_matrix(y_true, y_pred))

FIXED_PARAMS = {
    'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.2,
    'lr': 5e-4, 'weight_decay': 1e-4, 'activation': 'gelu', 'patience': 15,
}

def train_model(model_class, X_train, y_train, X_val, y_val,
                X_text_train, X_text_val, X_mask_train, X_mask_val, **params):
    def _ds(xn, xt, xm, y):
        return TensorDataset(
            torch.tensor(xn, dtype=torch.float32),
            torch.tensor(xt, dtype=torch.float32),
            torch.tensor(xm, dtype=torch.float32),
            torch.tensor(y,  dtype=torch.float32))
    train_loader = DataLoader(_ds(X_train,X_text_train,X_mask_train,y_train), shuffle=False, batch_size=32)
    val_loader   = DataLoader(_ds(X_val,  X_text_val,  X_mask_val,  y_val),   shuffle=False, batch_size=32)

    model = model_class(input_dim=X_train.shape[2],
                        hidden_dim=params.get('hidden_dim', 64),
                        num_layers=params.get('num_layers', 2),
                        dropout=params.get('dropout', 0.2),
                        activation=params.get('activation', 'gelu'))
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=params.get('lr', 5e-4),
                                 weight_decay=params.get('weight_decay', 1e-4))

    best_val_loss, best_state, no_improve = float('inf'), None, 0
    patience = params.get('patience', 15)

    for epoch in range(100):
        model.train()
        for bx, bt, bm, by in train_loader:
            optimizer.zero_grad()
            criterion(model(bx, bt, bm).view(-1), by).backward()
            optimizer.step()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for bx, bt, bm, by in val_loader:
                val_loss += criterion(model(bx, bt, bm).view(-1), by).item()
        avg = val_loss / len(val_loader)
        if avg < best_val_loss:
            best_val_loss, best_state, no_improve = avg, model.state_dict(), 0
        else:
            no_improve += 1
            if no_improve >= patience: break

    model.load_state_dict(best_state)
    return model


def run_cv_and_test(model_class, model_name, n_splits=5):
    """
    5-Fold TimeSeriesCV on train+val, then final evaluation on held-out test.
    Returns results dict.
    """
    print(f'{'='*60}')
    print(f'  {model_name}')
    print(f'{'='*60}')

    trainval_idx = np.concatenate([tr_idx, vl_idx])
    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_accs = []

    for fold, (cv_tr, cv_vl) in enumerate(tscv.split(trainval_idx), 1):
        a_tr = trainval_idx[cv_tr]
        a_vl = trainval_idx[cv_vl]

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_full_all[a_tr].reshape(-1, X_full_all.shape[2])).reshape(X_full_all[a_tr].shape)
        Xvl = scaler.transform(    X_full_all[a_vl].reshape(-1, X_full_all.shape[2])).reshape(X_full_all[a_vl].shape)

        m = train_model(model_class,
                        Xtr, y_full_all[a_tr], Xvl, y_full_all[a_vl],
                        X_text_all[a_tr], X_text_all[a_vl],
                        X_mask_all[a_tr], X_mask_all[a_vl],
                        **FIXED_PARAMS)
        m.eval()
        with torch.no_grad():
            preds = m(torch.tensor(Xvl, dtype=torch.float32),
                      torch.tensor(X_text_all[a_vl], dtype=torch.float32),
                      torch.tensor(X_mask_all[a_vl], dtype=torch.float32)).view(-1).numpy()
        fa = accuracy_score(y_full_all[a_vl], (preds > 0.5).astype(int))
        fold_accs.append(fa)
        print(f'  Fold {fold}: acc={fa:.4f}')

    print(f'  CV Mean Acc: {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}')

    # ── Final model on train+val, test on held-out ────────────────────────────
    print(f'\n  Training final model on train+val ...')
    scaler = StandardScaler()
    Xtv = scaler.fit_transform(X_full_all[trainval_idx].reshape(-1, X_full_all.shape[2])).reshape(X_full_all[trainval_idx].shape)
    Xte = scaler.transform(    X_full_all[te_idx].reshape(-1, X_full_all.shape[2])).reshape(X_full_all[te_idx].shape)

    final_m = train_model(model_class,
                          Xtv, y_full_all[trainval_idx], Xte, y_full_all[te_idx],
                          X_text_all[trainval_idx], X_text_all[te_idx],
                          X_mask_all[trainval_idx], X_mask_all[te_idx],
                          **FIXED_PARAMS)
    final_m.eval()
    with torch.no_grad():
        test_preds = final_m(
            torch.tensor(Xte, dtype=torch.float32),
            torch.tensor(X_text_all[te_idx], dtype=torch.float32),
            torch.tensor(X_mask_all[te_idx], dtype=torch.float32)).view(-1).numpy()

    y_test = y_full_all[te_idx]
    test_dates = np.array([pd.to_datetime(dates_full_all[i]) for i in te_idx])
    y_pred_bin, acc, prec_up, prec_down, rec_up, rec_down, f1, cm = evaluate_predictions(y_test, test_preds)

    results = {
        'y_true': y_test, 'y_pred': y_pred_bin, 'y_pred_prob': test_preds,
        'indices': test_dates, 'cm': cm,
        'acc': acc, 'prec_up': prec_up, 'rec_up': rec_up,
        'prec_down': prec_down, 'rec_down': rec_down, 'f1': f1,
        'cv_mean': np.mean(fold_accs), 'cv_std': np.std(fold_accs)
    }

    print(f'\n  TEST -> Accuracy: {acc:.4f} | F1: {f1:.4f} | Recall(Up): {rec_up:.4f}')

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(3, 2.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Down','Up'], yticklabels=['Down','Up'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'{model_name}\nConfusion Matrix', fontsize=9)
    plt.tight_layout(); plt.show()

    return results


def print_comparison_vs_baseline(results_dict, step_name):
    print(f'\n{"─"*65}')
    print(f'  {step_name} — Comparison vs Baseline')
    print(f'{"─"*65}')
    header = f'  {"Model":<40} {"Acc":>6}  {"F1":>6}  {"Rec(Up)":>8}'
    print(header)
    print('  ' + '-'*63)
    for name, b in BASELINES.items():
        print(f'  {name:<40} {b["acc"]:>6.4f}  {b["f1"]:>6.4f}  {b["rec_up"]:>8.4f}')
    print('  ' + '-'*63)
    for name, res in results_dict.items():
        delta_acc = res['acc'] - (BASELINES.get('Cross-Attention (baseline)', {}).get('acc', res['acc']))
        marker = '<<< NEW' if 'KASA' in name or 'KAN' in name or 'Sparse' in name else ''
        print(f'  {name:<40} {res["acc"]:>6.4f}  {res["f1"]:>6.4f}  {res["rec_up"]:>8.4f}  {marker}')
    print(f'{"─"*65}\n')

print('All shared utilities defined.')

---
## Step 1 — Sparsemax Projection (Noise Elimination)

**What changes:** In `AdditiveAttention`, replace `F.softmax` with `sparsemax` from the `entmax` library.

**Why it helps:** Sparsemax projects the raw score vector onto the probability simplex using Euclidean distance minimisation. Scores below the dynamic threshold $\tau(e_t)$ become exactly zero, mathematically severing irrelevant news days from the gradient pathway.

$$\alpha_{t,i} = \max(0, e_{t,i} - \tau(e_t))$$

Softmax attention is kept in `CrossAttentionLSTM` (`nn.MultiheadAttention`) — only the `AdditiveAttention` class that scores individual LSTM hidden states is upgraded.

In [ ]:
# ── Step 1: Sparsemax AdditiveAttention ──────────────────────────────────────

class SparseAdditiveAttention(nn.Module):
    """
    Bahdanau-style additive attention with Sparsemax normalisation.
    Replaces F.softmax(scores, dim=1) with sparsemax(scores, dim=1),
    producing sparse probability vectors that hard-zero noise entries.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 1))

    def forward(self, rnn_out):
        # rnn_out: (batch, seq_len, hidden_dim)
        scores  = self.attn(rnn_out)           # (batch, seq_len, 1)
        weights = sparsemax(scores, dim=1)     # sparse over time axis
        return torch.sum(weights * rnn_out, dim=1)  # (batch, hidden_dim)


# ── Step 1: CrossAttentionLSTM with Sparse Attention ─────────────────────────

class SparseCrossAttentionLSTM(nn.Module):
    uses_text = True

    def __init__(self, input_dim, text_dim=EMB_DIM, hidden_dim=64, num_layers=2,
                 dropout=0.2, activation='gelu'):
        super().__init__()
        self.num_rnn = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.text_rnn  = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.sentiment_rnn  = nn.LSTM(1, hidden_dim, 1, batch_first=True, dropout=0)
        # Sparse attention for the sentiment stream
        self.sentiment_attn = SparseAdditiveAttention(hidden_dim)

        self.cross_attn_price = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=4, dropout=dropout, batch_first=True)
        self.cross_attn_senti = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=4, dropout=dropout, batch_first=True)

        self.skip = GatedSkip(input_dim, hidden_dim)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim), get_activation(activation),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), get_activation(activation))
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim // 4), nn.LayerNorm(hidden_dim // 4),
            get_activation(activation), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1), nn.Sigmoid())

    def forward(self, x, x_text, x_mask):
        num_out, _ = self.num_rnn(x)
        t_proj = F.relu(self.text_proj(x_text))
        t_out, _ = self.text_rnn(t_proj)
        senti_input = x[:, :, -1].unsqueeze(-1)
        senti_out, _ = self.sentiment_rnn(senti_input)
        senti_context = self.sentiment_attn(senti_out)   # sparse

        cross_price, _ = self.cross_attn_price(t_out, num_out, num_out)
        cross_senti, _ = self.cross_attn_senti(t_out, senti_out, senti_out)
        context_price  = cross_price.mean(dim=1)
        context_senti  = cross_senti.mean(dim=1)

        combined = self.skip(context_price, x[:, -1, :])
        fused    = self.fusion(torch.cat([combined, context_senti, senti_context], dim=1))
        return self.fc(fused)


# ── Step 1: DualAttentionLSTM with Sparse Attention ───────────────────────────

class SparseDualAttentionLSTM(nn.Module):
    uses_text = True

    def __init__(self, input_dim, text_dim=EMB_DIM, hidden_dim=64, num_layers=2,
                 dropout=0.2, activation='gelu'):
        super().__init__()
        self.num_rnn  = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                                dropout=dropout if num_layers > 1 else 0)
        self.num_attn = SparseAdditiveAttention(hidden_dim)   # sparse
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.text_rnn  = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.text_attn = SparseAdditiveAttention(hidden_dim)  # sparse
        self.sentiment_rnn  = nn.LSTM(1, hidden_dim, 1, batch_first=True, dropout=0)
        self.sentiment_attn = SparseAdditiveAttention(hidden_dim)  # sparse

        self._gate_linear1 = nn.Linear(hidden_dim * 3 + 1 + 1, hidden_dim)
        self._gate_act     = get_activation(activation)
        self._gate_linear2 = nn.Linear(hidden_dim, 2)

        self.skip = GatedSkip(input_dim, hidden_dim)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim), get_activation(activation),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), get_activation(activation))
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim // 4), nn.LayerNorm(hidden_dim // 4),
            get_activation(activation), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1), nn.Sigmoid())

    def forward(self, x, x_text, x_mask):
        num_out, _  = self.num_rnn(x)
        num_context = self.num_attn(num_out)
        t_proj = F.relu(self.text_proj(x_text))
        t_out, _   = self.text_rnn(t_proj)
        t_context  = self.text_attn(t_out)
        senti_input = x[:, :, -1].unsqueeze(-1)
        senti_out, _ = self.sentiment_rnn(senti_input)
        senti_context = self.sentiment_attn(senti_out)

        global_mask   = x_mask.mean(dim=1).unsqueeze(1)
        sentiment_val = x[:, -1, -1].unsqueeze(1)
        fusion_input  = torch.cat([num_context, t_context, senti_context,
                                   global_mask, sentiment_val], dim=1)
        gate_hidden = self._gate_act(self._gate_linear1(fusion_input))
        gate = F.softmax(self._gate_linear2(gate_hidden), dim=1)

        combined            = self.skip(num_context, x[:, -1, :])
        combined_text_senti = t_context + senti_context
        gate[:, 0:1] * combined + gate[:, 1:2] * combined_text_senti  # weighted ctx (unused directly but computed for gate gradients)

        fused = self.fusion(torch.cat([combined, combined_text_senti, senti_context], dim=1))
        return self.fc(fused)


print('Step 1 modules (SparseAdditiveAttention, SparseCrossAttentionLSTM, SparseDualAttentionLSTM) defined.')

In [ ]:
step1_results = {}

step1_results['Step1: Sparse CrossAttention'] = run_cv_and_test(
    SparseCrossAttentionLSTM, 'Step1: Sparse CrossAttention LSTM')

In [ ]:
step1_results['Step1: Sparse DualAttention'] = run_cv_and_test(
    SparseDualAttentionLSTM, 'Step1: Sparse DualAttention LSTM')

In [ ]:
print_comparison_vs_baseline(step1_results, 'STEP 1 — Sparsemax')

---
## Step 2 — KAN Additive Attention (Non-Linear Similarity)

**What changes:** Replace the linear projection layers that compute the attention score with 1D learnable B-splines (KAN-style).

**Formulation:** The LSTM hidden state $h_t \in \mathbb{R}^{64}$ and the text/price context $x_i \in \mathbb{R}^{16}$ are concatenated into $z_{t,i} \in \mathbb{R}^{80}$. Each scalar dimension $d$ passes through an independent learnable spline $\Phi_d$:

$$e_{t,i} = \sum_{d=1}^{80} \Phi_d(z_{t,i,d}), \quad \Phi_d(u) = \sum_{n=1}^{N} c_{d,n} B_n(u)$$

**Normalisation:** Standard Softmax (to isolate the spline effect from sparsemax).

In [ ]:
# ── KANLinear: 1-D learnable B-splines ───────────────────────────────────────

class KANLinear(nn.Module):
    """
    Lightweight KAN layer using 1-D B-splines.

    For each input dimension d, maintains N learnable control points c_{d,n}.
    The output is the weighted sum of uniform B-spline basis functions evaluated
    at the (normalised) input value:

        output_d = sum_n  c_{d,n} * B_n(u_d)

    where B_n are triangular / uniform basis functions over [grid_min, grid_max].

    Total output = sum_d output_d   ->  scalar score per (h_t, x_i) pair.

    Args:
        in_features  : input dimensionality D (e.g. 80)
        n_knots      : number of B-spline control points N (default 5)
        grid_range   : value range for the spline grid (default [-1, 1])
    """
    def __init__(self, in_features: int, n_knots: int = 5,
                 grid_range: tuple = (-1., 1.)):
        super().__init__()
        self.in_features = in_features
        self.n_knots     = n_knots
        self.grid_min    = grid_range[0]
        self.grid_max    = grid_range[1]

        # Control points: (in_features, n_knots)  — learnable
        self.ctrl_pts = nn.Parameter(
            torch.zeros(in_features, n_knots).uniform_(-0.1, 0.1))

        # Fixed uniform knot positions: (n_knots,)
        self.register_buffer('knots',
            torch.linspace(grid_range[0], grid_range[1], n_knots))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., in_features)  — any leading batch dimensions
        returns: (...,)         — scalar per input vector
        """
        # Clamp to grid range for numerical stability
        x = x.clamp(self.grid_min, self.grid_max)   # (..., D)

        # B-spline basis: triangular hat functions
        # x unsqueezed: (..., D, 1)
        # knots:         (N,)
        h = (self.grid_max - self.grid_min) / max(self.n_knots - 1, 1)
        diff = x.unsqueeze(-1) - self.knots    # (..., D, N)
        basis = F.relu(1.0 - diff.abs() / h)   # hat functions; (..., D, N)

        # Weighted sum over knots: (..., D, N) * (D, N) -> (..., D)
        out_per_dim = (basis * self.ctrl_pts).sum(dim=-1)  # (..., D)

        # Sum over input dimensions -> scalar score
        return out_per_dim.sum(dim=-1)  # (...,)


# ── KANAdditiveAttention ──────────────────────────────────────────────────────

class KANAdditiveAttention(nn.Module):
    """
    KAN-based attention that concatenates [h_t | context_i] and scores
    the pair via independent B-splines on each dimension.

    When called with only rnn_out (no context), it falls back to a simple
    KAN projection of each hidden state to a scalar, exactly replacing
    the linear additive attention scorer used in the price/sentiment streams.

    Args:
        hidden_dim  : dimensionality of rnn_out  (e.g. 64)
        n_knots     : spline control points       (default 5)
    """
    def __init__(self, hidden_dim: int, n_knots: int = 5):
        super().__init__()
        self.scorer = KANLinear(hidden_dim, n_knots=n_knots)

    def forward(self, rnn_out: torch.Tensor) -> torch.Tensor:
        """
        rnn_out: (batch, seq_len, hidden_dim)
        returns: (batch, hidden_dim)  — weighted context vector
        """
        B, S, H = rnn_out.shape
        # Score each timestep: flatten (B, S, H) -> (B*S, H), score -> (B*S,)
        scores  = self.scorer(rnn_out.reshape(B * S, H)).reshape(B, S, 1)
        weights = F.softmax(scores, dim=1)  # Softmax for Step 2 (no sparsemax yet)
        return torch.sum(weights * rnn_out, dim=1)


print('KANLinear and KANAdditiveAttention defined.')
print(f'  KANLinear in_features=80, n_knots=5 -> {80*5} learnable spline params')

In [ ]:
# ── Step 2: KAN CrossAttentionLSTM ───────────────────────────────────────────

class KANCrossAttentionLSTM(nn.Module):
    uses_text = True

    def __init__(self, input_dim, text_dim=EMB_DIM, hidden_dim=64, num_layers=2,
                 dropout=0.2, activation='gelu', n_knots=5):
        super().__init__()
        self.num_rnn = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.text_rnn  = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.sentiment_rnn  = nn.LSTM(1, hidden_dim, 1, batch_first=True, dropout=0)
        # KAN attention for sentiment stream
        self.sentiment_attn = KANAdditiveAttention(hidden_dim, n_knots=n_knots)

        # Multi-head cross-attention kept standard (isolating KAN effect)
        self.cross_attn_price = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=4, dropout=dropout, batch_first=True)
        self.cross_attn_senti = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=4, dropout=dropout, batch_first=True)

        self.skip = GatedSkip(input_dim, hidden_dim)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim), get_activation(activation),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), get_activation(activation))
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim // 4), nn.LayerNorm(hidden_dim // 4),
            get_activation(activation), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1), nn.Sigmoid())

    def forward(self, x, x_text, x_mask):
        num_out, _ = self.num_rnn(x)
        t_proj = F.relu(self.text_proj(x_text))
        t_out, _ = self.text_rnn(t_proj)
        senti_input = x[:, :, -1].unsqueeze(-1)
        senti_out, _ = self.sentiment_rnn(senti_input)
        senti_context = self.sentiment_attn(senti_out)    # KAN scored

        cross_price, _ = self.cross_attn_price(t_out, num_out, num_out)
        cross_senti, _ = self.cross_attn_senti(t_out, senti_out, senti_out)
        context_price  = cross_price.mean(dim=1)
        context_senti  = cross_senti.mean(dim=1)

        combined = self.skip(context_price, x[:, -1, :])
        fused    = self.fusion(torch.cat([combined, context_senti, senti_context], dim=1))
        return self.fc(fused)


# ── Step 2: KAN DualAttentionLSTM ─────────────────────────────────────────────

class KANDualAttentionLSTM(nn.Module):
    uses_text = True

    def __init__(self, input_dim, text_dim=EMB_DIM, hidden_dim=64, num_layers=2,
                 dropout=0.2, activation='gelu', n_knots=5):
        super().__init__()
        self.num_rnn  = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                                dropout=dropout if num_layers > 1 else 0)
        self.num_attn = KANAdditiveAttention(hidden_dim, n_knots=n_knots)   # KAN
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.text_rnn  = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.text_attn = KANAdditiveAttention(hidden_dim, n_knots=n_knots)  # KAN
        self.sentiment_rnn  = nn.LSTM(1, hidden_dim, 1, batch_first=True, dropout=0)
        self.sentiment_attn = KANAdditiveAttention(hidden_dim, n_knots=n_knots)  # KAN

        self._gate_linear1 = nn.Linear(hidden_dim * 3 + 1 + 1, hidden_dim)
        self._gate_act     = get_activation(activation)
        self._gate_linear2 = nn.Linear(hidden_dim, 2)

        self.skip = GatedSkip(input_dim, hidden_dim)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim), get_activation(activation),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), get_activation(activation))
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim // 4), nn.LayerNorm(hidden_dim // 4),
            get_activation(activation), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1), nn.Sigmoid())

    def forward(self, x, x_text, x_mask):
        num_out, _  = self.num_rnn(x)
        num_context = self.num_attn(num_out)
        t_proj = F.relu(self.text_proj(x_text))
        t_out, _   = self.text_rnn(t_proj)
        t_context  = self.text_attn(t_out)
        senti_input = x[:, :, -1].unsqueeze(-1)
        senti_out, _ = self.sentiment_rnn(senti_input)
        senti_context = self.sentiment_attn(senti_out)

        global_mask   = x_mask.mean(dim=1).unsqueeze(1)
        sentiment_val = x[:, -1, -1].unsqueeze(1)
        fusion_input  = torch.cat([num_context, t_context, senti_context,
                                   global_mask, sentiment_val], dim=1)
        gate_hidden = self._gate_act(self._gate_linear1(fusion_input))
        gate = F.softmax(self._gate_linear2(gate_hidden), dim=1)

        combined            = self.skip(num_context, x[:, -1, :])
        combined_text_senti = t_context + senti_context
        gate[:, 0:1] * combined + gate[:, 1:2] * combined_text_senti

        fused = self.fusion(torch.cat([combined, combined_text_senti, senti_context], dim=1))
        return self.fc(fused)


print('Step 2 models (KANCrossAttentionLSTM, KANDualAttentionLSTM) defined.')

In [ ]:
step2_results = {}

step2_results['Step2: KAN CrossAttention'] = run_cv_and_test(
    KANCrossAttentionLSTM, 'Step2: KAN CrossAttention LSTM')

In [ ]:
step2_results['Step2: KAN DualAttention'] = run_cv_and_test(
    KANDualAttentionLSTM, 'Step2: KAN DualAttention LSTM')

In [ ]:
print_comparison_vs_baseline(step2_results, 'STEP 2 — KAN Additive Attention')

---
## Step 3 — Full KASA Architecture (Synthesis)

**What changes:** Combine the KAN B-spline scorer (Step 2) with Sparsemax normalisation (Step 1), and add an **asymmetric logit bias** of `+0.5` on the final pre-sigmoid output to explicitly protect `Recall (Up)` — the most important metric for long-side trading.

**Final attention weight formula:**
$$\alpha_{t,i} = \max\left(0,\ \left(\sum_{d=1}^{D} \Phi_d(h_{t,i,d})\right) - \tau(e_t)\right)$$

**Asymmetric bias:** The output layer includes a fixed `+0.5` shift to the logit before Sigmoid, pushing the decision boundary toward predicting "Up" more frequently, directly boosting recall on the positive class without retraining the entire network.

In [ ]:
# ── KAN + Sparsemax Attention ─────────────────────────────────────────────────

class KASAAttention(nn.Module):
    """
    Full KASA attention: KAN B-spline scorer + Sparsemax normalisation.

    Score:  e_{t,i} = KANLinear(h_{t,i})      — non-linear, per-dim splines
    Weight: α_{t,i} = sparsemax(e_t)           — exact-zero noise pruning
    Output: c_t     = Σ_i α_{t,i} · h_{t,i}   — sparse, non-linear context
    """
    def __init__(self, hidden_dim: int, n_knots: int = 5):
        super().__init__()
        self.scorer = KANLinear(hidden_dim, n_knots=n_knots)

    def forward(self, rnn_out: torch.Tensor) -> torch.Tensor:
        B, S, H = rnn_out.shape
        scores  = self.scorer(rnn_out.reshape(B * S, H)).reshape(B, S, 1)
        weights = sparsemax(scores, dim=1)  # sparse over time axis
        return torch.sum(weights * rnn_out, dim=1)


# ── Asymmetric Bias output wrapper ────────────────────────────────────────────

class AsymmetricBiasSigmoid(nn.Module):
    """
    Applies a fixed +bias shift to the logit before Sigmoid.
    Pushes decision boundary: P(Up) threshold = sigmoid(-bias) instead of 0.5.
    With bias=+0.5: threshold ~ 0.378  ->  model predicts Up more readily.
    """
    def __init__(self, bias: float = 0.5):
        super().__init__()
        self.bias = bias

    def forward(self, x):
        return torch.sigmoid(x + self.bias)


print('KASAAttention and AsymmetricBiasSigmoid defined.')
effective_threshold = torch.sigmoid(torch.tensor(-0.5)).item()
print(f'  AsymmetricBias=+0.5 -> effective Up threshold = {effective_threshold:.3f} (vs 0.500)')

In [ ]:
# ── Step 3: Full KASA CrossAttentionLSTM ──────────────────────────────────────

class KASACrossAttentionLSTM(nn.Module):
    uses_text = True

    def __init__(self, input_dim, text_dim=EMB_DIM, hidden_dim=64, num_layers=2,
                 dropout=0.2, activation='gelu', n_knots=5, asymmetric_bias=0.5):
        super().__init__()
        self.num_rnn = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.text_rnn  = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.sentiment_rnn  = nn.LSTM(1, hidden_dim, 1, batch_first=True, dropout=0)
        # Full KASA on sentiment stream
        self.sentiment_attn = KASAAttention(hidden_dim, n_knots=n_knots)

        self.cross_attn_price = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=4, dropout=dropout, batch_first=True)
        self.cross_attn_senti = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=4, dropout=dropout, batch_first=True)

        self.skip = GatedSkip(input_dim, hidden_dim)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim), get_activation(activation),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), get_activation(activation))
        # Final head with asymmetric bias
        self.fc_linear = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim // 4), nn.LayerNorm(hidden_dim // 4),
            get_activation(activation), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1))
        self.fc_out = AsymmetricBiasSigmoid(bias=asymmetric_bias)

    def forward(self, x, x_text, x_mask):
        num_out, _ = self.num_rnn(x)
        t_proj = F.relu(self.text_proj(x_text))
        t_out, _ = self.text_rnn(t_proj)
        senti_input = x[:, :, -1].unsqueeze(-1)
        senti_out, _ = self.sentiment_rnn(senti_input)
        senti_context = self.sentiment_attn(senti_out)   # KASA

        cross_price, _ = self.cross_attn_price(t_out, num_out, num_out)
        cross_senti, _ = self.cross_attn_senti(t_out, senti_out, senti_out)
        context_price  = cross_price.mean(dim=1)
        context_senti  = cross_senti.mean(dim=1)

        combined = self.skip(context_price, x[:, -1, :])
        fused    = self.fusion(torch.cat([combined, context_senti, senti_context], dim=1))
        return self.fc_out(self.fc_linear(fused))


# ── Step 3: Full KASA DualAttentionLSTM ───────────────────────────────────────

class KASADualAttentionLSTM(nn.Module):
    uses_text = True

    def __init__(self, input_dim, text_dim=EMB_DIM, hidden_dim=64, num_layers=2,
                 dropout=0.2, activation='gelu', n_knots=5, asymmetric_bias=0.5):
        super().__init__()
        self.num_rnn  = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                                dropout=dropout if num_layers > 1 else 0)
        # All three attention streams use full KASA
        self.num_attn  = KASAAttention(hidden_dim, n_knots=n_knots)
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.text_rnn  = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.text_attn = KASAAttention(hidden_dim, n_knots=n_knots)
        self.sentiment_rnn  = nn.LSTM(1, hidden_dim, 1, batch_first=True, dropout=0)
        self.sentiment_attn = KASAAttention(hidden_dim, n_knots=n_knots)

        self._gate_linear1 = nn.Linear(hidden_dim * 3 + 1 + 1, hidden_dim)
        self._gate_act     = get_activation(activation)
        self._gate_linear2 = nn.Linear(hidden_dim, 2)

        self.skip = GatedSkip(input_dim, hidden_dim)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim), get_activation(activation),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), get_activation(activation))
        # Final head with asymmetric bias
        self.fc_linear = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim // 4), nn.LayerNorm(hidden_dim // 4),
            get_activation(activation), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1))
        self.fc_out = AsymmetricBiasSigmoid(bias=asymmetric_bias)

    def forward(self, x, x_text, x_mask):
        num_out, _  = self.num_rnn(x)
        num_context = self.num_attn(num_out)
        t_proj = F.relu(self.text_proj(x_text))
        t_out, _   = self.text_rnn(t_proj)
        t_context  = self.text_attn(t_out)
        senti_input = x[:, :, -1].unsqueeze(-1)
        senti_out, _ = self.sentiment_rnn(senti_input)
        senti_context = self.sentiment_attn(senti_out)

        global_mask   = x_mask.mean(dim=1).unsqueeze(1)
        sentiment_val = x[:, -1, -1].unsqueeze(1)
        fusion_input  = torch.cat([num_context, t_context, senti_context,
                                   global_mask, sentiment_val], dim=1)
        gate_hidden = self._gate_act(self._gate_linear1(fusion_input))
        gate = F.softmax(self._gate_linear2(gate_hidden), dim=1)

        combined            = self.skip(num_context, x[:, -1, :])
        combined_text_senti = t_context + senti_context
        gate[:, 0:1] * combined + gate[:, 1:2] * combined_text_senti

        fused = self.fusion(torch.cat([combined, combined_text_senti, senti_context], dim=1))
        return self.fc_out(self.fc_linear(fused))


print('Step 3 KASA models defined.')
print('  KASACrossAttentionLSTM: KAN scorer + Sparsemax + AsymmetricBias(+0.5)')
print('  KASADualAttentionLSTM:  KAN scorer + Sparsemax + AsymmetricBias(+0.5) on ALL 3 streams')

In [ ]:
step3_results = {}

step3_results['Step3: KASA CrossAttention'] = run_cv_and_test(
    KASACrossAttentionLSTM, 'Step3: KASA CrossAttention LSTM')

In [ ]:
step3_results['Step3: KASA DualAttention'] = run_cv_and_test(
    KASADualAttentionLSTM, 'Step3: KASA DualAttention LSTM')

---
## Final Comparison — All Steps vs Baseline

In [ ]:
all_step_results = {**step1_results, **step2_results, **step3_results}

# ── Comprehensive summary table ───────────────────────────────────────────────
rows = {}
for name, b in BASELINES.items():
    rows[name] = {'Acc': b['acc'], 'F1': b['f1'], 'Rec(Up)': b['rec_up'],
                  'Prec(Up)': '—', 'Rec(Down)': '—', 'CV Mean': '—'}

for name, r in all_step_results.items():
    rows[name] = {
        'Acc':       round(r['acc'],    4),
        'F1':        round(r['f1'],     4),
        'Prec(Up)':  round(r['prec_up'],4),
        'Rec(Up)':   round(r['rec_up'], 4),
        'Rec(Down)': round(r['rec_down'],4),
        'CV Mean':   round(r.get('cv_mean', float('nan')), 4),
    }

df_all = pd.DataFrame(rows).T
print('\n' + '='*75)
print('  FULL KASA PROGRESSION — Test Set vs Baseline')
print('='*75)
print(df_all.to_string())

# Highlight best
numeric_cols = ['Acc', 'F1', 'Rec(Up)']
numeric_df = df_all[numeric_cols].apply(pd.to_numeric, errors='coerce')
print('\n  Best per metric (across all tested models):')
for col in numeric_cols:
    best = numeric_df[col].idxmax()
    print(f'    {col:12s}: {best}  ({numeric_df.loc[best, col]:.4f})')

In [ ]:
# ── Final confusion matrices for all Step 3 KASA models ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
fig.suptitle('Step 3 KASA — Final Confusion Matrices (Test Set)',
             fontsize=13, fontweight='bold')

for ax, (name, res) in zip(axes, step3_results.items()):
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Down','Up'], yticklabels=['Down','Up'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    short = name.replace('Step3: ', '')
    ax.set_title(f'{short}\nAcc={res["acc"]:.4f} | F1={res["f1"]:.4f} | Rec(Up)={res["rec_up"]:.4f}',
                 fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Bar chart: Acc / F1 / Rec(Up) progression across all steps ───────────────
plot_names = list(rows.keys())
acc_vals   = [float(rows[n]['Acc']) if rows[n]['Acc'] != '—' else np.nan for n in plot_names]
f1_vals    = [float(rows[n]['F1'])  if rows[n]['F1']  != '—' else np.nan for n in plot_names]
rec_up     = [float(rows[n]['Rec(Up)']) if rows[n]['Rec(Up)'] != '—' else np.nan for n in plot_names]

x = np.arange(len(plot_names))
width = 0.26

colors_baseline = ['#BBBBBB', '#BBBBBB']
colors_step     = ['#3498DB','#3498DB','#27AE60','#27AE60','#E74C3C','#E74C3C']
bar_colors      = colors_baseline + colors_step

fig, ax = plt.subplots(figsize=(14, 5))
b1 = ax.bar(x - width, acc_vals,  width, label='Accuracy',    color=[c for c in bar_colors], alpha=0.85)
b2 = ax.bar(x,         f1_vals,   width, label='F1 Score',    color=[c for c in bar_colors], alpha=0.65)
b3 = ax.bar(x + width, rec_up,    width, label='Recall (Up)', color=[c for c in bar_colors], alpha=0.5)

ax.set_xticks(x)
ax.set_xticklabels([n.replace('Step1: ','S1: ').replace('Step2: ','S2: ').replace('Step3: ','S3: ')
                    .replace('(baseline)','\n(base)') for n in plot_names],
                   rotation=20, ha='right', fontsize=9)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.4, label='50% baseline')
ax.set_ylim(0, 0.8)
ax.set_ylabel('Score')
ax.set_title('KASA Progression: Accuracy / F1 / Recall(Up) across all steps',
             fontsize=12, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#BBBBBB', label='Baseline (Softmax)'),
    Patch(facecolor='#3498DB', label='Step 1: Sparsemax'),
    Patch(facecolor='#27AE60', label='Step 2: KAN'),
    Patch(facecolor='#E74C3C', label='Step 3: KASA (KAN+Sparse+Bias)'),
]
ax.legend(handles=legend_elements + [plt.Line2D([0],[0],color='red',linestyle='--',label='50%')],
          loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Delta vs best baseline ────────────────────────────────────────────────────
print('\n' + '='*65)
print('  KASA FINAL REPORT — Delta vs Best Baseline')
print('='*65)

best_base_acc    = max(b['acc']    for b in BASELINES.values())
best_base_f1     = max(b['f1']     for b in BASELINES.values())
best_base_rec_up = max(b['rec_up'] for b in BASELINES.values())

print(f'  Best baseline: Acc={best_base_acc:.4f} | F1={best_base_f1:.4f} | Rec(Up)={best_base_rec_up:.4f}')
print()

for step_label, res_dict in [('Step 1 (Sparse)', step1_results),
                              ('Step 2 (KAN)',    step2_results),
                              ('Step 3 (KASA)',   step3_results)]:
    for name, r in res_dict.items():
        da = r['acc']    - best_base_acc
        df = r['f1']     - best_base_f1
        dr = r['rec_up'] - best_base_rec_up
        sign = lambda v: '+' if v >= 0 else ''
        short = name.split(': ', 1)[1]
        print(f'  [{step_label}] {short:<35}'
              f'  ΔAcc={sign(da)}{da:+.4f}  ΔF1={sign(df)}{df:+.4f}  ΔRec(Up)={sign(dr)}{dr:+.4f}')
    print()
print('='*65)